# 02 - BoT-SORT Two-stage Tuning

Notebook nay tune tracker theo 2 giai doan:
- `mot20`
- `hallway_rgbdt`

Notebook da duoc chinh de tan dung H100/system RAM lon bang cach tang sequence count va frame budget khi danh gia tracker config.


In [ ]:
REPO_URL = "https://github.com/ntdev204/adaptive-context-aware.git"
REPO_DIR = "/content/adaptive-context-aware"

In [ ]:
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!python -m pip install -e .[dev]
!pip install kagglehub

In [ ]:
import os
import shutil

import kagglehub

mot20_download = kagglehub.dataset_download("ismailelbouknify/mot-20")
hallway_download = kagglehub.dataset_download("kmader/hallway-rgbdt")

mot20_target = os.path.join(REPO_DIR, "data", "fine_tuning", "mot20")
hallway_target = os.path.join(REPO_DIR, "data", "fine_tuning", "hallway_rgbdt")

if os.path.exists(mot20_target):
    shutil.rmtree(mot20_target)
if os.path.exists(hallway_target):
    shutil.rmtree(hallway_target)

os.makedirs(os.path.dirname(mot20_target), exist_ok=True)
shutil.move(mot20_download, mot20_target)
shutil.move(hallway_download, hallway_target)
print("mot20:", mot20_target)
print("hallway_rgbdt:", hallway_target)

In [ ]:
import os

import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("cpu_count:", os.cpu_count())

In [ ]:
MAX_MOT20 = 4
MAX_HALLWAY = 4
MAX_FRAMES = 600
OUTPUT_JSON = "artifacts/tracker/botsort_tuned_h100.json"

In [ ]:
!python pipelines/train_tracker_botsort.py \
  --mot20-dir data/fine_tuning/mot20 \
  --hallway-rgbdt-dir data/fine_tuning/hallway_rgbdt \
  --output-path {OUTPUT_JSON} \
  --max-mot20-sequences {MAX_MOT20} \
  --max-hallway-sequences {MAX_HALLWAY} \
  --max-frames-per-sequence {MAX_FRAMES}

In [ ]:
!cat {OUTPUT_JSON}